In [1]:
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, and reporting of experiment
# results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `lmetk_report` package.
import sys
import pandas as pd
sys.path.append('..')

from lmetk_report.run_processor import load_runs_df
from lmetk_report.analysis import analyze_results
from lmetk_report.reporting import create_summary_table

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "TabPHM-autoencoders-checks_02_10_night"
USER = "imos-buildingenergy-1"
METRIC_PREFIXES = ["val/", "test/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse"]

# Configure pandas display options for better readability in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
df, config_columns, dropped_columns = load_runs_df(
    project_name=PROJECT_NAME, 
    user=USER
)

# ---
# ## 4. Core Analysis
# Run the main analysis function to group results by dataset and model,
# find the best hyperparameter configuration, and collect performance metrics.
# ---
all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES
)

# ---
# ## 5. Generate and Display Summary
# Create the final pivoted summary table for a high-level overview
# of the best performance for each model on each dataset.
# ---
summary_df = create_summary_table(all_results)

print("\n📊 Final Performance Summary Table:")
display(summary_df)



wandb: Currently logged in as: levsap to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded 56 runs from cached file: csv_files/TabPHM-autoencoders-checks_02_10_night.csv
Dropped 13 columns with >90% NaN values.
Dropped 12 columns based on name filters (e.g., 'paths.').
Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21 is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on UNIBO21:
optimizer.scheduler.scheduler_params.step_size: [10, 20]
optimizer.parameters.lr: [0.01, 0.001, 0.0005]
model.kernels: [[5, 5, 5, 5, 5], [7, 7, 5, 5, 5]]
model.strides: [[1, 1, 1, 1, 1], [2, 2, 1, 1, 1]]
model.dilations: [[1, 2, 4, 8, 1], [1, 1, 1, 1, 1]]
datamodule.val_batch_size: [16, 1024]
datamodule.test_batch_size: [16, 1024]
datamodule.train_batch_size: [16, 28, 128, 256, 512, 1024]
---------
Optimized metric for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA is val/loss
Hyperparameters that vary for lmetk.model.autoencoder.wrapper.AutoencoderWrapper on PRONOSTIA:
optimizer.scheduler.scheduler_pa

Dataset                                                      PRONOSTIA                                                                                                                          \
Metric                                                       test/loss            test/mae            test/mse           test/rmse            val/loss             val/mae             val/mse   
Model                                                                                                                                                                                            
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0001429764 ± nan  0.0073790648 ± nan  0.0002859528 ± nan  0.0119572747 ± nan  0.0001429764 ± nan  0.0073790648 ± nan  0.0002859528 ± nan   

Dataset                                                                            UNIBO21                                                                                                      \
Metric                                                        val/rmse           test/loss            test/mae            test/mse           test/rmse            val/loss             val/mae   
Model                                                                                                                                                                                            
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0119572747 ± nan  0.0000000919 ± nan  0.0001792862 ± nan  0.0000002756 ± nan  0.0003031131 ± nan  0.0000000919 ± nan  0.0001792862 ± nan   

Dataset                                                                                     
Metric                                                         val/mse            val/rmse  
Model                                                                                       
lmetk.model.autoencoder.wrapper.AutoencoderWrapper  0.0000002756 ± nan  0.0003031131 ± nan

In [5]:
for key in all_results.keys():
    print(f"\nDetailed results for dataset: {key}")
    for key_2 in all_results[key].keys():
        print(f"  Model: {key_2}")
        print("Available keys:", all_results[key][key_2].keys())
        print("Best hyperparameters:", all_results[key][key_2]["best_hyperparameters"])
        # print such that I can easily copy
        print("Best checkpoint path:\n", str(all_results[key][key_2]["best_checkpoint_path"]))
        display(all_results[key][key_2]["sorted_aggregated_results"].head())


Detailed results for dataset: UNIBO21
  Model: lmetk.model.autoencoder.wrapper.AutoencoderWrapper
Available keys: dict_keys(['best_hyperparameters', 'best_performance', 'best_run_names', 'best_checkpoint_path', 'avg_epoch_time', 'avg_runtime', 'sorted_aggregated_results', 'non_aggregated_df'])
Best hyperparameters: {'optimizer.scheduler.scheduler_params.step_size': 10, 'optimizer.parameters.lr': 0.0005, 'model.kernels': (5, 5, 5, 5, 5), 'model.strides': (1, 1, 1, 1, 1), 'model.dilations': (1, 2, 4, 8, 1), 'datamodule.val_batch_size': 1024, 'datamodule.test_batch_size': 1024, 'datamodule.train_batch_size': 256, 'run_names': ['unibo+reconstruction+residual_autoencoder_ea226fff']}
Best checkpoint path:
 /home/levtel/projects/TabPHM/artifacts/debug/runs/unibo+reconstruction+residual_autoencoder/2025-10-01_23-23-34/checkpoints/checkpoints


Model optimizer.scheduler.scheduler_params.step_size optimizer.parameters.lr    model.kernels    model.strides  model.dilations  \
                                                                                                                                                                                  
1   lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             10                  0.0005  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   
28  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             20                  0.0005  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   
8   lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             10                  0.0010  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   
36  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             20                  0.0010  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   
0   lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             10                  0.0005  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   

   datamodule.val_batch_size datamodule.test_batch_size datamodule.train_batch_size      val/loss           val/mse       val/mae      val/rmse         test/loss          test/mse      test/mae      \
                                                                                             mean std          mean std      mean std      mean std          mean std          mean std      mean std   
1                       1024                       1024                         256  9.187757e-08 NaN  2.756327e-07 NaN  0.000179 NaN  0.000303 NaN  9.187757e-08 NaN  2.756327e-07 NaN  0.000179 NaN   
28                      1024                       1024                         128  1.017384e-07 NaN  3.052154e-07 NaN  0.000179 NaN  0.000319 NaN  1.017384e-07 NaN  3.052154e-07 NaN  0.000179 NaN   
8                       1024                       1024                         128  1.390929e-07 NaN  4.172789e-07 NaN  0.000216 NaN  0.000373 NaN  1.390929e-07 NaN  4.172789e-07 NaN  0.000216 NaN   
36                      1024                       1024                         128  2.008883e-07 NaN  6.026648e-07 NaN  0.000279 NaN  0.000448 NaN  2.008883e-07 NaN  6.026648e-07 NaN  0.000279 NaN   
0                       1024                       1024                         128  2.394219e-07 NaN  7.182656e-07 NaN  0.000294 NaN  0.000489 NaN  2.394219e-07 NaN  7.182656e-07 NaN  0.000294 NaN   

   test/rmse                                               run_name  
        mean std                                               list  
1   0.000303 NaN  [unibo+reconstruction+residual_autoencoder_ea2...  
28  0.000319 NaN  [unibo+reconstruction+residual_autoencoder_942...  
8   0.000373 NaN  [unibo+reconstruction+residual_autoencoder_4ad...  
36  0.000448 NaN  [unibo+reconstruction+residual_autoencoder_4ee...  
0   0.000489 NaN  [unibo+reconstruction+residual_autoencoder_c47...


Detailed results for dataset: PRONOSTIA
  Model: lmetk.model.autoencoder.wrapper.AutoencoderWrapper
Available keys: dict_keys(['best_hyperparameters', 'best_performance', 'best_run_names', 'best_checkpoint_path', 'avg_epoch_time', 'avg_runtime', 'sorted_aggregated_results', 'non_aggregated_df'])
Best hyperparameters: {'optimizer.scheduler.scheduler_params.step_size': 10, 'model.kernels': (7, 7, 5, 5, 5), 'model.strides': (2, 2, 1, 1, 1), 'model.dilations': (1, 1, 1, 1, 1), 'run_names': ['pronostia+reconstruction+residual_autoencoder_6c4058ba']}
Best checkpoint path:
 /home/levtel/projects/TabPHM/artifacts/debug/runs/pronostia+reconstruction+residual_autoencoder/2025-10-02_03-30-38/checkpoints/checkpoints


Model optimizer.scheduler.scheduler_params.step_size    model.kernels    model.strides  model.dilations   val/loss        val/mse       val/mae      \
                                                                                                                                                             mean std       mean std      mean std   
1  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             10  (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)   0.000143 NaN   0.000286 NaN  0.007379 NaN   
0  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             10  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)   0.000143 NaN   0.000286 NaN  0.007391 NaN   
3  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             20  (7, 7, 5, 5, 5)  (2, 2, 1, 1, 1)  (1, 1, 1, 1, 1)   0.971066 NaN   1.942131 NaN  0.426614 NaN   
2  lmetk.model.autoencoder.wrapper.AutoencoderWra...                                             20  (5, 5, 5, 5, 5)  (1, 1, 1, 1, 1)  (1, 2, 4, 8, 1)  43.782490 NaN  87.564980 NaN  4.490829 NaN   

   val/rmse      test/loss       test/mse      test/mae     test/rmse                                               run_name  
       mean std       mean std       mean std      mean std      mean std                                               list  
1  0.011957 NaN   0.000143 NaN   0.000286 NaN  0.007379 NaN  0.011957 NaN  [pronostia+reconstruction+residual_autoencoder...  
0  0.011965 NaN   0.000143 NaN   0.000286 NaN  0.007391 NaN  0.011965 NaN  [pronostia+reconstruction+residual_autoencoder...  
3  0.985427 NaN   0.971066 NaN   1.942131 NaN  0.426614 NaN  0.985427 NaN  [pronostia+reconstruction+residual_autoencoder...  
2  6.616834 NaN  43.782490 NaN  87.564980 NaN  4.490829 NaN  6.616834 NaN  [pronostia+reconstruction+residual_autoencoder...

NameError: name 'subset' is not defined

In [3]:
df['callbacks.model_checkpoint.dirpath']

0     /home/levtel/projects/TabPHM/artifacts/debug/r...
1     /home/levtel/projects/TabPHM/artifacts/debug/r...
2     /home/levtel/projects/TabPHM/artifacts/debug/r...
3     /home/levtel/projects/TabPHM/artifacts/debug/r...
4     /home/levtel/projects/TabPHM/artifacts/debug/r...
5     /home/levtel/projects/TabPHM/artifacts/debug/r...
6     /home/levtel/projects/TabPHM/artifacts/debug/r...
7     /home/levtel/projects/TabPHM/artifacts/debug/r...
8     /home/levtel/projects/TabPHM/artifacts/debug/r...
9     /home/levtel/projects/TabPHM/artifacts/debug/r...
10    /home/levtel/projects/TabPHM/artifacts/debug/r...
11    /home/levtel/projects/TabPHM/artifacts/debug/r...
12    /home/levtel/projects/TabPHM/artifacts/debug/r...
13    /home/levtel/projects/TabPHM/artifacts/debug/r...
14    /home/levtel/projects/TabPHM/artifacts/debug/r...
15    /home/levtel/projects/TabPHM/artifacts/debug/r...
16    /home/levtel/projects/TabPHM/artifacts/debug/r...
17    /home/levtel/projects/TabPHM/artifacts/deb